# 03. Three-box ocean model  
# 手を動かして学ぶ 3-box 海洋炭素循環モデル  
# Hands-on three-box ocean carbon-cycle model

2-box では、表層を一つの箱として扱いました。  
In the 2-box model, the surface ocean was treated as one box.

3-box では、表層を **高緯度 H** と **低緯度 L** に分けます。  
In the 3-box model, the surface ocean is divided into **high-latitude H** and **low-latitude L**.

```text
Atmosphere
  ↑↓              ↑↓
High latitude H   Low latitude L
       \          /
          Deep D
```

この Notebook でも、**予想 → 実行 → 図 → 考察** を重視します。  
This notebook again emphasizes **prediction → execution → plotting → interpretation**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

plt.rcParams["figure.figsize"] = (7, 4)

## 1. なぜ 3-box にするのか / Why do we need three boxes?

2-box では、表層は一つでした。  
In the 2-box model, there was only one surface box.

しかし現実には、高緯度と低緯度は大きく違います。  
But in reality, high latitudes and low latitudes are very different.

```text
High latitude:
  cold
  deep-water formation
  ventilation

Low latitude:
  warm
  large surface area
  biological production
```

```text
高緯度:
  冷たい
  深層水形成
  ベンチレーション

低緯度:
  暖かい
  面積が大きい
  生物生産
```

この違いを表現するために、表層を H と L に分けます。  
To represent this contrast, we split the surface ocean into H and L.

In [ ]:
CV1 = 1.0250e3
CV2 = 9.7561e-4
CV3 = 1.0e6
CV4 = 3.1536e7
CV5 = 8.64e4

def to_umolkg(x):
    return CV2 * 1.0e6 * x

def to_ppmv(x):
    return CV3 * x

def o2sat(TEM, SAL):
    N1 = -1.734292e2
    N2 = +2.496339e2
    N3 = +1.433483e2
    N4 = -2.184920e1
    N5 = -3.309600e-2
    N6 = +1.425900e-2
    N7 = -1.700000e-3
    ATEM = TEM + 273.15
    O2S = math.exp(
        N1 + N2 * 1.0e2 / ATEM
        + N3 * math.log(ATEM / 1.0e2)
        + N4 * ATEM / 1.0e2
        + SAL * (N5 + N6 * ATEM / 1.0e2 + N7 * (ATEM / 1.0e2) ** 2)
    )
    return O2S * 4.35e1 * 1.025e-3

def chemeq_const(TEM, SAL):
    ATEM = TEM + 273.15
    TK = ATEM * 1.0e-2
    SK = 2.3517e-2 + (-2.3656e-2 + 4.7036e-3 * TK) * TK
    K0 = math.exp(-6.02409e1 + 9.34517e1 / TK + 2.33585e1 * math.log(TK) + SK * SAL)
    K1 = math.exp(math.log(10.0) * (
        13.7201 - 3.1334e-2 * ATEM - 3.23576e3 / ATEM
        - 1.3e-5 * SAL * ATEM + 1.032e-1 * math.sqrt(SAL)
    ))
    safe_sal = SAL if SAL > 0 else 1.0
    K2 = math.exp(math.log(10.0) * (
        -5.3719645e3 - 1.671221e0 * ATEM - 2.2913e-1 * SAL
        - 1.83802e1 * math.log10(safe_sal) + 1.2837528e5 / ATEM
        + 2.1943005e3 * math.log10(ATEM) + 8.0944e-4 * SAL * ATEM
        + 5.61711e3 * math.log10(safe_sal) / ATEM - 2.136e0 * SAL / ATEM
    ))
    KB = math.exp(math.log(10.0) * (-9.26 + 8.86e-3 * SAL + 1e-3 * TEM))
    KW = math.exp(
        +1.489802e2 - 1.384726e4 / ATEM - 2.36521e1 * math.log(ATEM)
        + ((-7.92447e1 + 3.29872e3 / ATEM + 1.20408e1 * math.log(ATEM))
           * math.sqrt(SAL) - 1.9813e-2 * SAL)
    )
    BT = 4.106e-4 * SAL / 35.0
    FH = 1.29e-2 - 2.4e-3 * ATEM + 4.61e-4 * SAL**2 - 1.48e-6 * ATEM * SAL**2
    return BT, K0, K1, K2, KB, KW, FH

def co2_nibun(BT, K0, K1, K2, KB, KW, AT, CT):
    ATX = CV2 * AT
    CTX = CV2 * CT
    HMIN = 1.0e-14
    HMAX = 1.0
    DELTA = 1.0e-15
    h_low = HMIN
    h_high = HMAX
    hx = 0.5 * (h_low + h_high)
    for _ in range(100000):
        h = 0.5 * (h_low + h_high)
        denom = h * h + K1 * h + K1 * K2
        at_calc = (
            (2.0 * K1 * K2 * CTX) / denom
            + (h * K1 * CTX) / denom
            + (KB * BT) / (h + KB)
            + KW / h
            - h
        )
        if abs(ATX - at_calc) <= DELTA:
            hx = h
            break
        if at_calc < ATX:
            h_high = h
        else:
            h_low = h
    denom2 = hx * hx + K1 * hx + K1 * K2
    PCO2 = (hx * hx * CTX) / denom2 / K0
    CO2 = (hx * hx * CTX) / denom2 / CV2
    HCO3 = K1 * CO2 / hx
    CO32 = K2 * HCO3 / hx
    return CO2, HCO3, CO32, PCO2

def carbonate_box(temp, sal, alk, dic):
    BT, K0, K1, K2, KB, KW, FH = chemeq_const(temp, sal)
    CO2, HCO3, CO32, PCO2 = co2_nibun(BT, K0, K1, K2, KB, KW, alk, dic)
    return {
        "BT": BT, "K0": K0, "K1": K1, "K2": K2,
        "KB": KB, "KW": KW, "CO2": CO2, "HCO3": HCO3,
        "CO32": CO32, "PCO2": PCO2
    }

## 2. ボックスの大きさ / Box geometry

高緯度 H は海洋面積の 15%、低緯度 L は 85% とします。  
The high-latitude box H occupies 15% of the ocean area, and the low-latitude box L occupies 85%.

H の厚さは 250 m、L の厚さは 100 m とします。  
The thickness is 250 m for H and 100 m for L.

深層 D は、全海洋体積から H と L を引いた残りです。  
The deep box D is the remaining volume after subtracting H and L from the total ocean volume.

In [ ]:
VOCN = 1.292e18
AOCN = 3.49e14
VATM = 1.773e20

AOCNH = AOCN * 0.15
AOCNL = AOCN * 0.85

ZOCNH = 250.0
ZOCNL = 100.0

VOCNH = AOCNH * ZOCNH
VOCNL = AOCNL * ZOCNL
VOCND = VOCN - VOCNH - VOCNL

pd.DataFrame({
    "Box": ["H high-latitude surface", "L low-latitude surface", "D deep"],
    "Area fraction": [0.15, 0.85, np.nan],
    "Volume [m3]": [VOCNH, VOCNL, VOCND],
})

### Mini exercise 1 / ミニ演習 1

低緯度 L の厚さを 50 m, 100 m, 200 m に変えると、L の体積はどう変わりますか。  
Change the thickness of L to 50 m, 100 m, and 200 m. How does the volume of L change?

これは「ボックスの厚さ」が物質量に直接効くことを確認する演習です。  
This exercise checks that box thickness directly affects the amount of material in the box.

In [ ]:
for z in [50, 100, 200]:
    vocnl = AOCNL * z
    print(f"ZOCNL = {z:3.0f} m  VOCNL = {vocnl:.3e} m3")

## 3. 温度と初期値 / Temperature and initial values

H は冷たく、L は暖かく、D は深層です。  
H is cold, L is warm, and D is the deep ocean.

ここでも、まずは普通の値として変数を入れます。  
Again, we first store values as ordinary variables.

`PO4H` は H box の PO4、`PO4L` は L box の PO4、`PO4D` は D box の PO4 です。  
`PO4H`, `PO4L`, and `PO4D` are phosphate in H, L, and D boxes.

In [ ]:
TEMH = 2.0
TEML = 21.5
TEMD = 1.75
SALH = 34.7
SALL = 34.7
SALD = 34.7

def initial_three_box():
    return {
        "PO4H": 2.10e-6 * CV1,
        "PO4L": 2.10e-6 * CV1,
        "PO4D": 2.10e-6 * CV1,
        "ALKH": 2.374e-3 * CV1,
        "ALKL": 2.374e-3 * CV1,
        "ALKD": 2.374e-3 * CV1,
        "DICH": 2.235e-3 * CV1,
        "DICL": 2.235e-3 * CV1,
        "DICD": 2.235e-3 * CV1,
        "DO2H": 1.60e-4 * CV1,
        "DO2L": 1.60e-4 * CV1,
        "DO2D": 1.60e-4 * CV1,
        "PCO2A": 280.0 / CV3,
    }

x = initial_three_box()

pd.DataFrame({
    "Tracer": ["PO4", "DIC", "ALK", "O2"],
    "H": [to_umolkg(x["PO4H"]), to_umolkg(x["DICH"]), to_umolkg(x["ALKH"]), to_umolkg(x["DO2H"])],
    "L": [to_umolkg(x["PO4L"]), to_umolkg(x["DICL"]), to_umolkg(x["ALKL"]), to_umolkg(x["DO2L"])],
    "D": [to_umolkg(x["PO4D"]), to_umolkg(x["DICD"]), to_umolkg(x["ALKD"]), to_umolkg(x["DO2D"])],
})

## 4. 同じ DIC/ALK でも pCO2 は違う / pCO2 differs even with the same DIC and ALK

H と L は温度が違います。  
H and L have different temperatures.

そのため、同じ DIC と ALK でも pCO2 が変わります。  
Therefore, even with the same DIC and ALK, pCO2 differs.

これは、海洋炭素循環で温度が重要な理由の一つです。  
This is one reason why temperature is important in ocean carbon cycling.

In [ ]:
carbH = carbonate_box(TEMH, SALH, x["ALKH"], x["DICH"])
carbL = carbonate_box(TEML, SALL, x["ALKL"], x["DICL"])
carbD = carbonate_box(TEMD, SALD, x["ALKD"], x["DICD"])

pd.DataFrame({
    "Box": ["H", "L", "D"],
    "Temperature [deg C]": [TEMH, TEML, TEMD],
    "pCO2 [ppmv]": [to_ppmv(carbH["PCO2"]), to_ppmv(carbL["PCO2"]), to_ppmv(carbD["PCO2"])],
})

## 5. パラメタ / Parameters

3-box では、次の交換を考えます。  
The 3-box model includes the following exchanges.

```text
T   : large-scale circulation
FHD : H-D exchange
FLD : L-D exchange
FLH : L-H exchange
```

特に `FHD` は、深層ベンチレーションの強さに関係します。  
In particular, `FHD` is related to deep-ocean ventilation.

In [ ]:
DT = 8.64e4
T = 2.0e7
FHD = 6.0e7
FLD = 0.0
FLH = 0.0

PVH = 3.0
PVL = 3.0
FAH = PVH * AOCNH / CV5
FAL = PVL * AOCNL / CV5

RCP = 106.0
RNP = 16.0
RRC = 0.20
RO2P = 172.0

CEPH = 0.75
EPH = (CEPH / RCP) * AOCNH / CV4

R = 1.0 / CV4
LF = 0.5
DEL = 100.0
HSC = 2.0e-8 * CV1

pd.DataFrame({
    "Parameter": ["T", "FHD", "FLD", "FLH", "FAH", "FAL", "EPH"],
    "Value": [T, FHD, FLD, FLH, FAH, FAL, EPH],
})

## 6. EPL は毎ステップ変わる / EPL changes every time step

ここが 3-box の重要点です。  
This is a key point of the 3-box model.

低緯度の輸出生産 `EPL` は、低緯度リン酸 `PO4L` に依存します。  
Low-latitude export `EPL` depends on low-latitude phosphate `PO4L`.

```text
EPL = R * DEL * LF * PO4L * PO4L / (HSC + PO4L) * VOCNL
```

したがって、`PO4L` が時間変化すると `EPL` も時間変化します。  
Therefore, when `PO4L` evolves in time, `EPL` also evolves in time.

これは固定パラメタではなく、**診断されるフラックス**です。  
It is not a fixed parameter, but a **diagnosed flux**.

In [ ]:
def compute_epl(x, LF=0.5):
    return R * DEL * LF * x["PO4L"] * (x["PO4L"] / (HSC + x["PO4L"])) * VOCNL

EPL = compute_epl(x)

print("EPH [PgC/yr] =", CV4 * EPH * RCP * 12.0e-15)
print("EPL [PgC/yr] =", CV4 * EPL * RCP * 12.0e-15)

### Mini exercise 2 / ミニ演習 2

`PO4L` を半分にすると `EPL` は半分になるでしょうか。  
If `PO4L` is halved, does `EPL` become half?

この式には `PO4L` が 2 回出てくるので、単純な比例ではありません。  
Since `PO4L` appears twice in the formula, the relationship is not simply linear.

In [ ]:
x_test = dict(x)
x_test["PO4L"] = 0.5 * x["PO4L"]

EPL_original = compute_epl(x)
EPL_half_po4 = compute_epl(x_test)

print("Original EPL =", EPL_original)
print("Half PO4L EPL =", EPL_half_po4)
print("Ratio =", EPL_half_po4 / EPL_original)

## 7. PO4 を 1 日だけ進める / Advance PO4 by one day

まず PO4 だけ見ます。  
First, we look only at PO4.

```text
H: L と D から入る、EPH で減る
L: D と H から入る、EPL で減る
D: H と L から入る、再無機化で増える
```

```text
H: receives from L and D, loses by EPH
L: receives from D and H, loses by EPL
D: receives from H and L, gains by remineralization
```

このセルは、Fortran の式を Python に移したときの基本形でもあります。  
This cell also shows the basic form of translating the Fortran equation into Python.

In [ ]:
EPL = compute_epl(x)

PO4HX = x["PO4H"] + ((T + FLH) * (x["PO4L"] - x["PO4H"]) + FHD * (x["PO4D"] - x["PO4H"]) - EPH) * (DT / VOCNH)
PO4LX = x["PO4L"] + ((T + FLD) * (x["PO4D"] - x["PO4L"]) + FLH * (x["PO4H"] - x["PO4L"]) - EPL) * (DT / VOCNL)
PO4DX = x["PO4D"] + ((T + FHD) * (x["PO4H"] - x["PO4D"]) + FLD * (x["PO4L"] - x["PO4D"]) + (EPH + EPL)) * (DT / VOCND)

pd.DataFrame({
    "Box": ["H", "L", "D"],
    "Old PO4": [to_umolkg(x["PO4H"]), to_umolkg(x["PO4L"]), to_umolkg(x["PO4D"])],
    "New PO4": [to_umolkg(PO4HX), to_umolkg(PO4LX), to_umolkg(PO4DX)],
})

## 8. 1 ステップ関数 / One-step function

ここで、PO4, ALK, DIC, O2, 大気 CO2 をまとめて 1 日進める関数を作ります。  
Now we define a function that advances PO4, ALK, DIC, O2, and atmospheric CO2 by one day.

関数の入力は現在の値、出力は 1 日後の値です。  
The input is the current values, and the output is the values after one day.

```text
current values
↓
one_step_three_box()
↓
next values
```

`x` や `y` は特別なものではありません。  
`x` and `y` are not special concepts.

ここでは、`x` が現在の値、`y` が次の値を入れる袋です。  
Here, `x` is a container for current values and `y` is a container for next values.

In [ ]:
def one_step_three_box(x, T=2.0e7, FHD=6.0e7, FLD=0.0, FLH=0.0,
                       CEPH=0.75, LF=0.5, air_sea=True):
    EPH = (CEPH / RCP) * AOCNH / CV4
    EPL = compute_epl(x, LF=LF)

    carbH = carbonate_box(TEMH, SALH, x["ALKH"], x["DICH"])
    carbL = carbonate_box(TEML, SALL, x["ALKL"], x["DICL"])

    alk_factor = 2.0 * RRC * RCP - RNP
    dic_factor = (1.0 + RRC) * RCP

    y = dict(x)

    y["PO4H"] = x["PO4H"] + ((T + FLH) * (x["PO4L"] - x["PO4H"]) + FHD * (x["PO4D"] - x["PO4H"]) - EPH) * (DT / VOCNH)
    y["PO4L"] = x["PO4L"] + ((T + FLD) * (x["PO4D"] - x["PO4L"]) + FLH * (x["PO4H"] - x["PO4L"]) - EPL) * (DT / VOCNL)
    y["PO4D"] = x["PO4D"] + ((T + FHD) * (x["PO4H"] - x["PO4D"]) + FLD * (x["PO4L"] - x["PO4D"]) + EPH + EPL) * (DT / VOCND)

    y["ALKH"] = x["ALKH"] + ((T + FLH) * (x["ALKL"] - x["ALKH"]) + FHD * (x["ALKD"] - x["ALKH"]) - alk_factor * EPH) * (DT / VOCNH)
    y["ALKL"] = x["ALKL"] + ((T + FLD) * (x["ALKD"] - x["ALKL"]) + FLH * (x["ALKH"] - x["ALKL"]) - alk_factor * EPL) * (DT / VOCNL)
    y["ALKD"] = x["ALKD"] + ((T + FHD) * (x["ALKH"] - x["ALKD"]) + FLD * (x["ALKL"] - x["ALKD"]) + alk_factor * (EPH + EPL)) * (DT / VOCND)

    gasH = 0.0
    gasL = 0.0
    if air_sea:
        gasH = FAH * CV1 * carbH["K0"] * (x["PCO2A"] - carbH["PCO2"])
        gasL = FAL * CV1 * carbL["K0"] * (x["PCO2A"] - carbL["PCO2"])

    y["DICH"] = x["DICH"] + ((T + FLH) * (x["DICL"] - x["DICH"]) + FHD * (x["DICD"] - x["DICH"]) - dic_factor * EPH + gasH) * (DT / VOCNH)
    y["DICL"] = x["DICL"] + ((T + FLD) * (x["DICD"] - x["DICL"]) + FLH * (x["DICH"] - x["DICL"]) - dic_factor * EPL + gasL) * (DT / VOCNL)
    y["DICD"] = x["DICD"] + ((T + FHD) * (x["DICH"] - x["DICD"]) + FLD * (x["DICL"] - x["DICD"]) + dic_factor * (EPH + EPL)) * (DT / VOCND)

    O2SATH = o2sat(TEMH, SALH)
    O2SATL = o2sat(TEML, SALL)

    y["DO2H"] = x["DO2H"] + ((T + FLH) * (O2SATL - x["DO2H"]) + FHD * (O2SATH - x["DO2H"]) + RO2P * EPH + FAH * (O2SATH - x["DO2H"])) * (DT / VOCNH)
    y["DO2L"] = x["DO2L"] + ((T + FLD) * (x["DO2D"] - x["DO2L"]) + FLH * (x["DO2H"] - x["DO2L"]) + RO2P * EPL + FAL * (O2SATL - x["DO2L"])) * (DT / VOCNL)
    y["DO2D"] = x["DO2D"] + ((T + FHD) * (x["DO2H"] - x["DO2D"]) + FLD * (O2SATL - x["DO2D"]) - RO2P * (EPH + EPL)) * (DT / VOCND)

    if air_sea:
        carbH_new = carbonate_box(TEMH, SALH, y["ALKH"], y["DICH"])
        carbL_new = carbonate_box(TEML, SALL, y["ALKL"], y["DICL"])
        y["PCO2A"] = x["PCO2A"] + (
            CV1 * carbH_new["K0"] * FAH * (carbH_new["PCO2"] - x["PCO2A"])
            + CV1 * carbL_new["K0"] * FAL * (carbL_new["PCO2"] - x["PCO2A"])
        ) * (DT / VATM)

    diagnostics = {"EPH": EPH, "EPL": EPL}
    return y, diagnostics

## 9. 時間積分 / Time integration

1 ステップ関数を繰り返し呼び出します。  
We repeatedly call the one-step function.

これにより、1 日後、2 日後、...、3000 年後の状態が計算されます。  
This gives the state after 1 day, 2 days, ..., and 3000 years.

1 年ごとに値を保存して、時系列として図にします。  
We save values once per year and plot them as time series.

In [ ]:
def run_three_box(years=3000, T=2.0e7, FHD=6.0e7, FLD=0.0, FLH=0.0,
                  CEPH=0.75, LF=0.5, air_sea=True):
    x = initial_three_box()
    history = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            carbH = carbonate_box(TEMH, SALH, x["ALKH"], x["DICH"])
            carbL = carbonate_box(TEML, SALL, x["ALKL"], x["DICL"])
            carbD = carbonate_box(TEMD, SALD, x["ALKD"], x["DICD"])
            _, diag = one_step_three_box(x, T=T, FHD=FHD, FLD=FLD, FLH=FLH, CEPH=CEPH, LF=LF, air_sea=air_sea)
            history.append({
                "year": day / 365,
                "PO4H": to_umolkg(x["PO4H"]),
                "PO4L": to_umolkg(x["PO4L"]),
                "PO4D": to_umolkg(x["PO4D"]),
                "DICH": to_umolkg(x["DICH"]),
                "DICL": to_umolkg(x["DICL"]),
                "DICD": to_umolkg(x["DICD"]),
                "DO2H": to_umolkg(x["DO2H"]),
                "DO2L": to_umolkg(x["DO2L"]),
                "DO2D": to_umolkg(x["DO2D"]),
                "PCO2H": to_ppmv(carbH["PCO2"]),
                "PCO2L": to_ppmv(carbL["PCO2"]),
                "PCO2D": to_ppmv(carbD["PCO2"]),
                "PCO2A": to_ppmv(x["PCO2A"]),
                "EPH_PgCyr": CV4 * diag["EPH"] * RCP * 12.0e-15,
                "EPL_PgCyr": CV4 * diag["EPL"] * RCP * 12.0e-15,
            })
        x, diag = one_step_three_box(x, T=T, FHD=FHD, FLD=FLD, FLH=FLH, CEPH=CEPH, LF=LF, air_sea=air_sea)

    return x, pd.DataFrame(history)

final, hist = run_three_box()
hist.tail()

## 10. 図で見る / Plot the results

3-box では、H, L, D の違いを見ることが大事です。  
In the 3-box model, it is important to compare H, L, and D.

特に見てほしい点は次です。  
Pay attention to:

- H と L の PO4 の違い  
  difference in PO4 between H and L

- 深層 O2 の変化  
  change in deep O2

- 大気 pCO2 と H/L pCO2 の関係  
  relationship among atmospheric pCO2 and H/L pCO2

In [ ]:
for var, ylabel in [
    ("PO4", "PO4 [umol/kg]"),
    ("DIC", "DIC [umol/kg]"),
    ("DO2", "O2 [umol/kg]"),
]:
    plt.figure()
    plt.plot(hist["year"], hist[f"{var}H"], label="H")
    plt.plot(hist["year"], hist[f"{var}L"], label="L")
    plt.plot(hist["year"], hist[f"{var}D"], label="D")
    plt.xlabel("Time [yr]")
    plt.ylabel(ylabel)
    plt.title(var)
    plt.legend()
    plt.grid(True)
    plt.show()

plt.figure()
plt.plot(hist["year"], hist["PCO2A"], label="Atmosphere")
plt.plot(hist["year"], hist["PCO2H"], label="H")
plt.plot(hist["year"], hist["PCO2L"], label="L")
plt.xlabel("Time [yr]")
plt.ylabel("pCO2 [ppmv]")
plt.title("pCO2")
plt.legend()
plt.grid(True)
plt.show()

## 11. Challenge A: 深層ベンチレーション FHD を変える / Change deep ventilation FHD

`FHD` は H と D の交換です。  
`FHD` is the exchange between H and D.

`FHD` を小さくすると、深層はより孤立します。  
If `FHD` is reduced, the deep ocean becomes more isolated.

**予想 / Prediction**

`FHD` を小さくすると、深層 O2 は増えるでしょうか、減るでしょうか。  
If `FHD` is reduced, does deep O2 increase or decrease?

In [ ]:
FHD_list = [1.0e7, 2.0e7, 4.0e7, 6.0e7, 8.0e7, 1.0e8]
summary = []

plt.figure()
for f in FHD_list:
    _, h = run_three_box(years=3000, FHD=f)
    plt.plot(h["year"], h["DO2D"], label=f"FHD={f:.1e}")
    summary.append({
        "FHD": f,
        "Final O2D": h["DO2D"].iloc[-1],
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final PO4H": h["PO4H"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Deep O2 [umol/kg]")
plt.title("Sensitivity to FHD")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

## 12. Challenge B: 低緯度生物ポンプ LF を変える / Change low-latitude biological pump LF

`LF` は低緯度の輸出生産 `EPL` の強さに関係します。  
`LF` controls the strength of low-latitude export `EPL`.

`LF` を大きくすると、生物ポンプが強くなります。  
A larger `LF` means a stronger biological pump.

**予想 / Prediction**

`LF` を大きくすると、低緯度 PO4 と大気 pCO2 はどう変わるでしょうか。  
If `LF` is increased, what happens to low-latitude PO4 and atmospheric pCO2?

In [ ]:
LF_list = [0.2, 0.5, 0.8, 1.2]
summary = []

plt.figure()
for lf in LF_list:
    _, h = run_three_box(years=3000, LF=lf)
    plt.plot(h["year"], h["EPL_PgCyr"], label=f"LF={lf}")
    summary.append({
        "LF": lf,
        "Final EPL": h["EPL_PgCyr"].iloc[-1],
        "Final PO4L": h["PO4L"].iloc[-1],
        "Final pCO2A": h["PCO2A"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("EPL [PgC/yr]")
plt.title("Sensitivity of low-latitude export")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

## 13. Challenge C: H と L をつなぐ FLH / Connect H and L with FLH

標準では `FLH = 0` です。  
In the standard case, `FLH = 0`.

`FLH` を入れると、高緯度表層 H と低緯度表層 L が直接混ざります。  
If `FLH` is nonzero, high-latitude surface H and low-latitude surface L mix directly.

**予想 / Prediction**

`FLH` を大きくすると、H と L の差は大きくなるでしょうか、小さくなるでしょうか。  
If `FLH` is increased, does the contrast between H and L become larger or smaller?

In [ ]:
FLH_list = [0.0, 1.0e7, 2.0e7, 4.0e7]
summary = []

plt.figure()
for flh in FLH_list:
    _, h = run_three_box(years=3000, FLH=flh)
    plt.plot(h["year"], h["PO4H"], label=f"FLH={flh:.1e}")
    summary.append({
        "FLH": flh,
        "Final PO4H": h["PO4H"].iloc[-1],
        "Final PO4L": h["PO4L"].iloc[-1],
        "Final pCO2A": h["PCO2A"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("High-latitude PO4 [umol/kg]")
plt.title("Sensitivity to H-L exchange")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

## 14. 自分で実験する / Your own experiment

ここは自由実験セルです。  
This is a free experiment cell.

まずは次を変えてみてください。  
Try changing these first:

```python
my_FHD
my_FLH
my_LF
my_air_sea
```

実験するときは、必ず「予想」を先に書いてから実行してください。  
When you run an experiment, write your prediction before executing the code.

In [ ]:
my_T = 2.0e7
my_FHD = 6.0e7
my_FLD = 0.0
my_FLH = 0.0
my_LF = 0.5
my_air_sea = True

final_my, hist_my = run_three_box(
    years=3000,
    T=my_T,
    FHD=my_FHD,
    FLD=my_FLD,
    FLH=my_FLH,
    LF=my_LF,
    air_sea=my_air_sea,
)

plt.figure()
plt.plot(hist_my["year"], hist_my["PCO2A"], label="Atmosphere")
plt.plot(hist_my["year"], hist_my["PCO2H"], label="H")
plt.plot(hist_my["year"], hist_my["PCO2L"], label="L")
plt.xlabel("Time [yr]")
plt.ylabel("pCO2 [ppmv]")
plt.title("My experiment")
plt.legend()
plt.grid(True)
plt.show()

## 15. 課題 / Exercises

### 課題 1 / Exercise 1

`FHD` を小さくすると、深層 O2 と大気 pCO2 はどう変わるか。  
What happens to deep O2 and atmospheric pCO2 when `FHD` is reduced?

### 課題 2 / Exercise 2

`LF` を大きくすると、低緯度 PO4 と EPL はどう変わるか。  
What happens to low-latitude PO4 and EPL when `LF` is increased?

### 課題 3 / Exercise 3

`FLH` を 0 から増やすと、H と L の差はどう変わるか。  
What happens to the contrast between H and L when `FLH` is increased?

### 課題 4 / Exercise 4

2-box と比較して、3-box で新しく表現できる現象を説明せよ。  
Explain what new phenomena can be represented in the 3-box model compared with the 2-box model.